# 01 · The data and the core idea

The whole project rests on one property of this dataset: **the reference is exact by construction**. Turns are sampled first, features are rendered from them, so there is no annotation error anywhere in the evaluation. This notebook shows that, then shows why the embedder has something to learn.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
os.environ.setdefault("OMP_NUM_THREADS", "2")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch; torch.set_num_threads(2)

TABLES = os.path.join("..", "results", "tables")
def table(name):
    return pd.read_csv(os.path.join(TABLES, name))
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


## A generated conversation

Features on top, the exact reference below. The turn boundaries are where the generator put them, not where an annotator guessed.

In [ ]:
from streamdiar.config import load_config
from streamdiar.data.generator import (
    dataset_stats, generate_recording, generate_split,
)
cfg = load_config('../configs/base.yaml')
rec = generate_recording(cfg.data, 0, 1000, 'test')
print(rec.name, rec.features.shape,
      f'{rec.n_speakers} speakers, {len(rec.turns)} turns')
print(f'overlap {rec.overlap_fraction():.1%} of speech frames, '
      f'speech {rec.speech_fraction():.1%} of all frames')

In [ ]:
ref = rec.reference_matrix()
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
im = axes[0].imshow(rec.features.T, aspect='auto', origin='lower', cmap='magma',
                    extent=(0, rec.duration_s, 0, rec.features.shape[1]))
axes[0].set_ylabel('feature channel'); axes[0].set_title(rec.name)
fig.colorbar(im, ax=axes[0], pad=0.01, label='log power')
t = np.arange(rec.n_frames) / rec.frame_rate
for s in range(rec.n_speakers):
    axes[1].fill_between(t, s, s + 0.8, where=ref[:, s], step='mid',
                         color=plt.cm.tab10(s % 10), alpha=0.85)
axes[1].set_yticks(np.arange(rec.n_speakers) + 0.4)
axes[1].set_yticklabels([f'spk {s}' for s in range(rec.n_speakers)])
axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('reference')
plt.tight_layout(); plt.show()

## Overlap really is summed in the power domain

Two people talking at once combine as `log(exp(a) + exp(b))`, not `a + b`. If it were the latter, overlap would be a trivially detectable amplitude spike and the overlap experiments would be meaningless. Doubling the power adds only `log 2 ≈ 0.69`.

In [ ]:
counts = ref.sum(axis=1)
rows = []
for k, name in [(0, 'silence'), (1, 'one speaker'), (2, 'two speakers')]:
    sel = counts == k if k < 2 else counts >= 2
    if sel.any():
        rows.append({'region': name, 'frames': int(sel.sum()),
                     'mean log power': float(rec.features[sel].mean())})
disp = pd.DataFrame(rows)
print(disp.to_string(index=False))
print('\nlog(2) =', round(float(np.log(2)), 4),
      '<- the most a second equal-power talker can add')

## Split statistics

The test split is 24 recordings of 60 s, 2–5 speakers each, drawn from a speaker pool disjoint from training.

In [ ]:
stats = dataset_stats(generate_split(cfg.data, 0, 'test'))
print(pd.Series(stats).to_string())

## Why the thresholds are near 0.85, not 0.55

This is the figure that explains every threshold in the config — and the mistake that motivated measuring it. The embedding space is a narrow cone, so a cosine-similarity intuition of "0.55 means different speakers" is badly wrong. Guessing 0.55 made the diarizer find 1.62 speakers where there were 3.50.

In [ ]:
from streamdiar.pipelines.common import dev_split, load_or_train
from streamdiar.pipelines.tuning import precompute, similarity_distributions
os.chdir('..')  # checkpoints/ and results/ are repo-relative
model = load_or_train(cfg, 0, verbose=False)
cached = precompute(model, cfg, dev_split(cfg, 0))
same, diff = similarity_distributions(cached, cfg.frames_per_hop)
os.chdir('notebooks')
bins = np.linspace(-1, 1, 80)
plt.figure(figsize=(9, 4))
plt.hist(diff, bins=bins, alpha=0.6, density=True,
         label='different speakers', color='#e45756')
plt.hist(same, bins=bins, alpha=0.6, density=True,
         label='same speaker', color='#4c78a8')
plt.axvline(np.median(same), color='#4c78a8', ls='--',
         label=f'same median {np.median(same):.3f}')
plt.axvline(np.median(diff), color='#e45756', ls='--',
         label=f'diff median {np.median(diff):.3f}')
plt.axvline(0.83, color='k', ls=':', lw=2, label='tuned spawn_threshold 0.83')
plt.xlabel('cosine similarity between window embeddings'); plt.ylabel('density')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Causality, as a guarantee rather than a statistic

Perturb every frame after `t0` and the embeddings of all windows ending at or before `t0` are **bit-identical**. Not close — equal. The non-causal variant fails the same check, which is what makes the clean result meaningful.

In [ ]:
from streamdiar.config import replace
from streamdiar.models.embedder import build_embedder
x = np.random.RandomState(0).randn(2000, cfg.data.n_features).astype(np.float32)
x2 = x.copy(); t0 = 1200
x2[t0:] = np.random.RandomState(1).randn(2000 - t0, cfg.data.n_features)
rows = []
for causal in [True, False]:
    m = build_embedder(replace(cfg.embedder, causal=causal), cfg.data.n_features,
                       model.feature_mean, model.feature_sd, seed=0)
    a, _, ends = m.embed_recording(x, cfg.frames_per_window, cfg.frames_per_hop)
    b, _, _ = m.embed_recording(x2, cfg.frames_per_window, cfg.frames_per_hop)
    early = ends <= t0
    rows.append({'causal': causal, 'windows checked': int(early.sum()),
                 'max abs difference': float(np.abs(a[early] - b[early]).max())})
print(pd.DataFrame(rows).to_string(index=False))